# LogiOn — 음성 업무 비서, 한 문장이 처리되는 과정

편의점에 물건을 납품하는 트럭 기사가 운전하거나 짐을 내리면서 **손을 쓰지 않고 말로** 업무를 처리하도록 만든 비서입니다.
"다음 어디야", "우유 두 개 터졌어" 같은 한국어 한 문장을 받아서, 그 말이 어떤 기능인지 알아내고, 필요한 값을 뽑아내고,
배송 DB 에 실제로 반영한 다음, 읽어 줄 문장을 만듭니다. 모든 처리는 **보드(Orange Pi 5+) 안에서** 도는 것을 목표로 합니다.

이 노트북은 **학습 코드가 없습니다.** 이미 만들어진 모델을 불러와서 **추론만** 하고, 한 문장이 어떤 단계를 거치는지 단계마다 펼쳐서 보여 줍니다.
(모델을 만드는 노트북은 [`LogiOn_AI_pipeline.ipynb`](LogiOn_AI_pipeline.ipynb) 입니다.)

![LogiOn 구조](../docs/architecture_v2_1.png)

## 한 문장이 거치는 단계

| 단계 | 하는 일 | 쓰는 것 |
| --- | --- | --- |
| ① 입력 | 기사가 말한 문장 (지금은 텍스트, 나중에 STT) | — |
| ② 의도분류 | 6개 기능 중 무엇인지 + 얼마나 확신하는지 | klue/roberta-small |
| ③ function calling | 문장에서 값을 뽑아 JSON 한 줄로 | Qwen2.5-1.5B + LoRA |
| ④ 값 보정 | 말한 이름을 DB 이름으로 맞추기, 모자란 값은 되묻기 | 규칙 + DB |
| ⑤ 실행 | 배송 DB 에 반영 (백엔드 API) | FastAPI + SQLite |
| ⑥ 응답 | 읽어 줄 문장 만들기 (나중에 TTS) | 규칙 |

**②에서 바로 ⑤로 건너뛰기도 합니다.** `scan`·`other` 처럼 값이 필요 없는 기능은 의도분류가 충분히 확신하면 LLM 을 부르지 않습니다.
반대로 1등과 2등 확률이 비슷하면(**margin** 이 작으면) 실행하지 않고 **두 후보를 말해 주며 되묻습니다.**

### 6개 기능

| 기능 | 언제 | 값 |
| --- | --- | --- |
| `next_stop` | 다음·남은 배송지를 물을 때 | 없음 |
| `stop_items` | 내릴 품목·수량을 물을 때 | `store?`, `item?` |
| `complete` | 납품을 마쳤다고 알릴 때 | `store?`, `receiver?` |
| `report_issue` | 물건에 문제가 있다고 알릴 때 | `store?`, `item?`, `issue`, `qty?`, `unit?` |
| `scan` | 바코드를 찍겠다고 할 때 | 없음 |
| `other` | 업무와 상관없는 말 | 없음 |

## 0. 설정 불러오기

기준값(되묻는 문턱 `tau`, LLM 을 건너뛰는 확신 기준)은 **`config/pipeline_v2_1.json`** 한 파일에 들어 있습니다.
이 값들은 **val 로만** 골랐고 test 는 마지막에 한 번만 봤습니다.

In [1]:
import json, os, sys, time, difflib, re
from pathlib import Path

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
CFG = json.loads((ROOT / "config/pipeline_v2_1.json").read_text(encoding="utf-8"))

INTENT_DIR = ROOT / CFG["intent_model"]
FC_DIR = ROOT / CFG["fc_model"]
missing = [str(p) for p in (INTENT_DIR, FC_DIR) if not (p / "config.json").exists()]
READY = not missing

print("설정:", CFG["version"], "| tau =", CFG["ask_tau"],
      "| LLM 건너뛰기:", CFG["skip_llm_fns"], ">=", CFG["skip_llm_conf"])
print("의도분류 모델:", INTENT_DIR)
print("function calling 모델:", FC_DIR)
if missing:
    print("\n[모델 없음]", ", ".join(missing))
    print("먼저 notebooks/LogiOn_AI_pipeline.ipynb 를 실행해 모델을 만드세요. (GPU 필요, 약 40분)")
else:
    print("\n모델 파일 확인됨.")

설정: v2.1 | tau = 0.05 | LLM 건너뛰기: ['scan', 'other'] >= 0.9
의도분류 모델: ..\out\intent_model
function calling 모델: ..\out\fc_merged

모델 파일 확인됨.


In [2]:
if not READY:
    raise SystemExit("모델이 없어 여기서 멈춥니다. 먼저 notebooks/LogiOn_AI_pipeline.ipynb 를 실행해 모델을 만드세요.")

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("장치:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

t0 = time.time()
intent_tok = AutoTokenizer.from_pretrained(INTENT_DIR)
intent_model = AutoModelForSequenceClassification.from_pretrained(INTENT_DIR).eval().to(DEVICE)
intent_model.config.torchscript = False
FUNCS = CFG["funcs"]
ID2LABEL = {i: f for i, f in enumerate(FUNCS)}

fc_tok = AutoTokenizer.from_pretrained(FC_DIR)
if fc_tok.pad_token is None:
    fc_tok.pad_token = fc_tok.eos_token
fc_model = AutoModelForCausalLM.from_pretrained(
    FC_DIR, torch_dtype=DTYPE, device_map=({"": 0} if DEVICE == "cuda" else None)).eval()
if DEVICE == "cpu":
    fc_model = fc_model.to("cpu")
print(f"모델 불러오기 {time.time() - t0:.0f}초")

장치: cuda NVIDIA GeForce RTX 5050


The tokenizer you are loading from '..\out\fc_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


`torch_dtype` is deprecated! Use `dtype` instead!


모델 불러오기 2초


## 1. 배송 데이터 (백엔드와 같은 것)

값 보정("우유" → 어떤 우유인지)은 **실제 DB 이름**을 기준으로 해야 합니다.
그래서 백엔드의 초기 데이터 **`backend/app/db/seed.py`** 를 그대로 읽어 씁니다. (백엔드 코드는 읽기만 하고 고치지 않습니다.)

`SEND_TO_BACKEND = False` 이면 이 노트북 안의 가상 DB 로 결과를 만들고,
`True` 이고 서버가 살아 있으면 **기존 API 를 그대로 호출**합니다.

In [3]:
SEND_TO_BACKEND = False                      # True 로 바꾸면 아래 주소의 기존 API 를 실제로 호출
BACKEND_URL = "http://127.0.0.1:8000"

sys.path.insert(0, str((ROOT / "backend").resolve()))
from app.db import seed as SEED             # 백엔드 초기 데이터 (읽기만 함)

STORES = [{"id": i, "name": n, "short_name": sn} for i, n, sn, *_ in SEED.STORES]
PRODUCTS = [{"id": i, "name": n, "category": c, "aliases": sorted({f.replace(" ", "") for f in forms})}
            for i, (n, (c, forms)) in enumerate(SEED.PRODUCTS.items(), start=1)]
PID = {p["name"]: p["id"] for p in PRODUCTS}

# 오늘 운행 (seed.ROUTE_STOPS 와 같은 내용을 메모리에 올린 가상 DB)
STOPS = []
for seq, (store_id, eta, km, notice, zones, items) in enumerate(SEED.ROUTE_STOPS, start=1):
    store = next(s for s in STORES if s["id"] == store_id)
    STOPS.append({"id": seq, "seq": seq, "store": store, "eta": eta, "status": "pending",
                  "receiver": None,
                  "items": [{"product_id": PID[n], "name": n, "qty": q, "unit": u} for n, q, u in items],
                  "issues": []})


def next_stop():
    # 다음 배송지 = 아직 안 끝난 곳 중 순서가 가장 빠른 곳 (delivery._next_stop_id 와 같은 규칙)
    return next((s for s in STOPS if s["status"] == "pending"), None)


print("점포", len(STORES), "/ 품목", len(PRODUCTS), "/ 오늘 배송지", len(STOPS))
for s in STOPS:
    goods = ", ".join("{} {}{}".format(i["name"], i["qty"], i["unit"]) for i in s["items"])
    print(f"  {s['seq']}. {s['store']['name']} ({s['store']['short_name']}) - {goods}")
print("\n다음 배송지:", next_stop()["store"]["name"])

점포 3 / 품목 20 / 오늘 배송지 3
  1. GS25 성수연무장점 (성수연무장점) - 서울우유 12개, 바나나우유 6개, 제육도시락 4개, 생수 2박스
  2. CU 서울숲점 (서울숲점) - 서울우유 6개, 참치삼각김밥 10개, 콜라 1박스
  3. 세븐일레븐 뚝섬점 (뚝섬점) - 샌드위치 8개, 캔커피 2박스, 초코파이 3박스

다음 배송지: GS25 성수연무장점


## 2. 각 단계를 하는 함수

### ② 의도분류
6개 기능의 확률을 내고, **1등과 2등의 차이(margin)** 로 세 갈래 중 하나를 고릅니다.

### ③ function calling
LLM 이 JSON 한 줄을 만듭니다. **확신도**는 생성한 토큰들의 확률에서 평균과 최솟값을 봅니다.
(최솟값이 낮으면 그 한 글자를 자신 없게 찍은 것이라, 값 하나가 틀렸을 가능성이 높습니다.)

### ④ 값 보정
말한 이름을 DB 이름에 맞춥니다. **정확 → 별칭 → 포함 → 발음 유사도** 순서로 찾고,
후보가 여러 개면 실행하지 않고 되묻습니다.

In [4]:
@torch.no_grad()
def classify(text):
    enc = intent_tok([text], truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
    p = intent_model(**enc).logits.float().softmax(-1)[0].cpu().numpy()
    order = np.argsort(-p)
    return p, [(ID2LABEL[int(i)], float(p[i])) for i in order]


def decide(ranked):
    # ② 의도분류 결과 -> 어느 길로 갈지. 기준값은 config 에서 읽은 것
    (f1, p1), (f2, p2) = ranked[0], ranked[1]
    margin = p1 - p2
    if margin < CFG["ask_tau"]:
        if "other" in (f1, f2):
            return {"route": "ask", "opts": [], "msg": CFG["retry_message"], "margin": margin,
                    "why": f"margin {margin:.3f} < {CFG['ask_tau']} 이고 후보에 other 가 있어 선택지를 만들 수 없음"}
        L = CFG["choice_labels"]
        return {"route": "ask", "opts": [f1, f2], "margin": margin,
                "msg": f"{L[f1]} 말씀하시는 건가요, {L[f2]} 말씀하시는 건가요?",
                "why": f"margin {margin:.3f} < {CFG['ask_tau']} 이라 1등을 믿을 수 없음"}
    if f1 in CFG["skip_llm_fns"] and p1 >= CFG["skip_llm_conf"]:
        return {"route": "skip", "fn": f1, "margin": margin,
                "why": f"{f1} 은 값이 필요 없고 확률 {p1:.3f} >= {CFG['skip_llm_conf']} 이라 LLM 생략"}
    return {"route": "llm", "fn": f1, "margin": margin,
            "why": (f"{f1} 은 값을 뽑아야 하므로 LLM 으로" if f1 not in CFG["skip_llm_fns"]
                    else f"{f1} 이지만 확률 {p1:.3f} < {CFG['skip_llm_conf']} 이라 LLM 으로 확인")}


def build_prompt(text):
    return fc_tok.apply_chat_template(
        [{"role": "system", "content": CFG["system_prompt"]}, {"role": "user", "content": text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)


@torch.no_grad()
def call_llm(text):
    # ③ LLM 출력 문자열과 토큰 확률(평균·최솟값)
    enc = fc_tok([build_prompt(text)], return_tensors="pt", add_special_tokens=False).to(fc_model.device)
    out = fc_model.generate(**enc, max_new_tokens=CFG["generation"]["max_new_tokens"], do_sample=False,
                            pad_token_id=fc_tok.pad_token_id, return_dict_in_generate=True, output_scores=True)
    new = out.sequences[0, enc["input_ids"].shape[1]:]
    probs = [float(torch.softmax(s[0].float(), -1)[t]) for s, t in zip(out.scores, new)]
    raw = fc_tok.decode(new, skip_special_tokens=True).strip()
    return raw, (float(np.mean(probs)) if probs else 0.0), (float(np.min(probs)) if probs else 0.0)


def parse_json(raw):
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return {"_bad": True}
    try:
        d = json.loads(m.group(0))
        return d if isinstance(d, dict) else {"_bad": True}
    except Exception:
        return {"_bad": True}

In [5]:
def _flat(s):
    return str(s).replace(" ", "")


def match_name(spoken, rows, name_keys, alias_key=None, cutoff=0.6):
    # 말한 이름 -> DB 행. (후보들, 찾은 방법, 점수) 를 함께 돌려줘 화면에 보여 준다
    q = _flat(spoken)
    hits = [r for r in rows if any(_flat(r[k]) == q for k in name_keys)]
    if hits:
        return hits, "정확", 1.0
    if alias_key:
        hits = [r for r in rows if q in r[alias_key]]
        if hits:
            return hits, "별칭", 1.0
    hits = [r for r in rows if any(q in _flat(r[k]) or _flat(r[k]) in q for k in name_keys)]
    if hits:
        return hits, "포함", 1.0
    scored = [(max(difflib.SequenceMatcher(None, q, _flat(r[k])).ratio() for k in name_keys), r) for r in rows]
    scored.sort(key=lambda x: -x[0])
    if scored and scored[0][0] >= cutoff:
        best = scored[0][0]
        return [r for s, r in scored if s >= best - 1e-9], "발음 유사도", best
    return [], "못 찾음", (scored[0][0] if scored else 0.0)


def resolve(call, steps):
    # ④ 값 보정. steps 에 '무엇을 어떻게 바꿨는지' 를 쌓아서 화면에 보여 준다
    # 돌려주는 것: (보정된 call, 되물을 말 or None)
    out = dict(call)
    out.pop("_bad", None)
    fn = out.get("fn")

    # 점포: 말했으면 DB 에서 찾고, 안 말했으면 다음 배송지
    if out.get("store"):
        rows, how, score = match_name(out["store"], STORES, ["name", "short_name"])
        cands = [r["name"] for r in rows]
        steps.append({"값": "store", "말한 것": out["store"],
                      "찾은 것": (cands[0] if len(rows) == 1 else "—"), "방법": how,
                      "점수": round(score, 3), "후보": cands})
        if len(rows) == 1:
            stop = next((s for s in STOPS if s["store"]["id"] == rows[0]["id"]), None)
            if stop is None:
                return out, f"{rows[0]['name']} 은 오늘 배송지에 없어요."
            out["_stop"] = stop
        elif len(rows) > 1:
            return out, "어느 점포 말씀이신가요? " + ", ".join(cands)
        else:
            return out, f"'{out['store']}' 이라는 점포를 찾지 못했어요."
    else:
        stop = next_stop()
        steps.append({"값": "store", "말한 것": "(말하지 않음)",
                      "찾은 것": (stop["store"]["name"] if stop else "—"),
                      "방법": "다음 배송지로 채움", "점수": "—", "후보": []})
        if stop is None and fn in ("stop_items", "complete", "report_issue"):
            return out, "오늘 배송이 모두 끝났어요."
        out["_stop"] = stop

    # 품목
    if out.get("item"):
        rows, how, score = match_name(out["item"], PRODUCTS, ["name"], alias_key="aliases")
        cands = [r["name"] for r in rows]
        steps.append({"값": "item", "말한 것": out["item"],
                      "찾은 것": (cands[0] if len(rows) == 1 else "—"), "방법": how,
                      "점수": round(score, 3), "후보": cands})
        if len(rows) == 1:
            out["item"], out["product_id"] = rows[0]["name"], rows[0]["id"]
        elif len(rows) > 1:
            return out, ", ".join(cands) + " 중 어떤 건가요?"
        else:
            return out, f"'{out['item']}' 이라는 품목을 찾지 못했어요."

    # 단위: 네 가지 중 하나로 맞춤 (백엔드 CHECK 제약과 같은 목록)
    if out.get("qty") is not None:
        UNITS = ["개", "박스", "판", "봉지"]
        u = out.get("unit")
        if u not in UNITS:
            rows, how, score = ([{"name": u}], "정확", 1.0) if u in UNITS else ([], "", 0)
            new = "개" if not u else min(UNITS, key=lambda x: -difflib.SequenceMatcher(None, u, x).ratio())
            steps.append({"값": "unit", "말한 것": (u or "(말하지 않음)"), "찾은 것": new,
                          "방법": ("기본값 개" if not u else "가장 가까운 단위"), "점수": "—", "후보": UNITS})
            out["unit"] = new

    # 기능마다 꼭 있어야 하는 값이 비었으면 그것만 되묻는다
    if fn == "report_issue":
        if not out.get("item"):
            return out, "어떤 품목인가요?"
        if not out.get("issue"):
            return out, "어떤 문제인가요? 파손, 수량부족, 오배송, 온도이상 중에서 말씀해 주세요."
    return out, None

## 3. ⑤ 실행 — 백엔드에 보낼 요청

`SEND_TO_BACKEND = False` (기본) 이면 **보낼 요청만 보여 주고** 노트북 안 가상 DB 로 결과를 만듭니다.
`True` 이고 서버가 살아 있으면 **기존 API 를 그대로 호출**합니다. 어느 쪽이든 **백엔드 코드는 고치지 않습니다.**

`report_issue` 는 보내기 전에 두 가지를 바꿔야 합니다 (`backend/app/api/routes.py` 의 `IssueIn` 기준).
- `issue` → **`type`** 으로 이름 바꾸기
- 품목 **이름 → `product_id`** (숫자)

In [6]:
def build_request(fn, call):
    # fn + 보정된 값 -> 백엔드에 보낼 요청 (메서드, 경로, 본문)
    stop = call.get("_stop")
    sid = stop["id"] if stop else None
    if fn == "next_stop":
        return {"method": "GET", "path": "/api/routes/today", "body": None}
    if fn == "stop_items":
        return {"method": "GET", "path": f"/api/stops/{sid}", "body": None}
    if fn == "complete":
        return {"method": "POST", "path": f"/api/stops/{sid}/complete",
                "body": {"receiver": call.get("receiver")}}
    if fn == "report_issue":
        body = {"type": call.get("issue"), "source": "voice"}      # issue -> type
        if call.get("product_id"):
            body["product_id"] = call["product_id"]                 # 이름 -> 숫자
        if call.get("qty") is not None:
            body["qty"], body["unit"] = call["qty"], call.get("unit")
        return {"method": "POST", "path": f"/api/stops/{sid}/issues", "body": body}
    return {"method": None, "path": None, "body": None}             # scan / other 는 DB 를 바꾸지 않음


def send_real(req):
    import requests
    url = BACKEND_URL.rstrip("/") + req["path"]
    r = (requests.get(url, timeout=5) if req["method"] == "GET"
         else requests.post(url, json=req["body"], timeout=5))
    r.raise_for_status()
    return r.json()


def run_virtual(fn, call, req):
    # 노트북 안 가상 DB 로 같은 일을 한다 (백엔드를 안 띄웠을 때)
    stop = call.get("_stop")
    if fn == "next_stop":
        left = [s for s in STOPS if s["status"] == "pending"]
        return {"next_stop_id": (left[0]["id"] if left else None),
                "remaining": len(left),
                "next_store": (left[0]["store"]["name"] if left else None)}
    if fn == "stop_items":
        items = stop["items"]
        if call.get("item"):
            items = [i for i in items if i["name"] == call["item"]]
        return {"stop_id": stop["id"], "store": stop["store"]["name"], "items": items}
    if fn == "complete":
        stop["status"], stop["receiver"] = "done", call.get("receiver")
        return {"stop_id": stop["id"], "store": stop["store"]["name"], "status": "done",
                "receiver": stop["receiver"]}
    if fn == "report_issue":
        issue = dict(req["body"], id=len(stop["issues"]) + 1, stop_id=stop["id"],
                     product_name=call.get("item"))
        stop["issues"].append(issue)
        return issue
    if fn == "scan":
        return {"action": "open_scanner"}
    return {}

In [7]:
def josa(word, with_batchim, without_batchim):
    # 받침 유무에 따라 조사를 고른다 (콜라"는" / 서울우유"는" / 참치김밥"은")
    if not word:
        return without_batchim
    ch = word[-1]
    if "가" <= ch <= "힣":
        return with_batchim if (ord(ch) - 0xAC00) % 28 else without_batchim
    return without_batchim


def make_reply(fn, call, result, send):
    # ⑥ 읽어 줄 문장. 가상 DB 든 실제 API 든 같은 모양의 결과를 받아서 만든다
    if fn == "next_stop":
        if send:
            left = [s for s in result["stops"] if s["status"] == "pending"]
            return (f"다음은 {left[0]['store']['name']}이에요. 남은 곳은 {len(left)}곳이에요."
                    if left else "오늘 배송이 모두 끝났어요.")
        return (f"다음은 {result['next_store']}이에요. 남은 곳은 {result['remaining']}곳이에요."
                if result["next_store"] else "오늘 배송이 모두 끝났어요.")
    if fn == "stop_items":
        items = result["items"]
        store = result["store"] if not send else result["store"]["name"]
        if call.get("item"):
            items = [i for i in items if i["name"] == call["item"]]
            if not items:
                return f"{store}에 {call['item']}{josa(call['item'], '은', '는')} 없어요."
            i = items[0]
            return f"{store} {i['name']}{josa(i['name'], '은', '는')} {i['qty']}{i['unit']}예요."
        if not items:
            return f"{store}에 내릴 물건이 없어요."
        return f"{store} 납품은 " + ", ".join(f"{i['name']} {i['qty']}{i['unit']}" for i in items) + "예요."
    if fn == "complete":
        store = result["store"] if not send else result["store"]["name"]
        r = result.get("receiver")
        return f"{store} 납품 완료로 기록했어요." + (f" 인수자는 {r}이에요." if r else "")
    if fn == "report_issue":
        store = call["_stop"]["store"]["name"]
        qty = f" {result['qty']}{result['unit']}" if result.get("qty") else ""
        return f"{store} {result.get('product_name') or '품목 미상'}{qty} {result['type']}으로 기록했어요."
    if fn == "scan":
        return "스캔 화면을 열게요."
    return "죄송해요, 배송 업무만 도와드릴 수 있어요."

## 4. 전체를 잇는 `run(text)`

되묻는 중이면 그 답을 이어받습니다. `PENDING` 에 "무엇을 왜 물었는지" 를 들고 있다가,
다음 말이 그 답이면 거기에 이어서 처리하고, 답이 아니면 **새로운 문장으로 보고 처음부터** 합니다.

In [8]:
PENDING = {}            # {"kind": "slot"/"choice", ...}
ORD = {0: ["첫", "1번", "하나", "처음", "앞"], 1: ["두", "2번", "둘", "뒤", "나중"]}


def match_choice(text):
    # 선택지 되묻기에 대한 답을 알아본다. 못 알아보면 None
    if PENDING.get("kind") != "choice":
        return None
    t = text.replace(" ", "")
    KW = {"next_stop": ["배송지", "다음", "남은곳", "행선지"], "stop_items": ["물건", "품목", "내릴", "물량"],
          "report_issue": ["문제", "이상", "보고", "파손"], "complete": ["완료", "납품완료", "끝"],
          "scan": ["스캔", "바코드", "찍"]}
    for fn in PENDING["opts"]:
        if any(k in t for k in KW.get(fn, [])):
            return fn
    for i, words in ORD.items():
        if any(w in t for w in words) and i < len(PENDING["opts"]):
            return PENDING["opts"][i]
    return None


def run(text, send=None, show=True):
    send = SEND_TO_BACKEND if send is None else send
    T, t_all = {}, time.perf_counter()
    out = {"text": text}
    if show:
        print("=" * 100)
        print(f"① 입력   🗣  {text}")

    # 되묻기 답 이어받기 -----------------------------------------------------
    picked = match_choice(text)
    carry = None
    if picked:
        carry, text_for_llm = ("choice", PENDING["text"])
        if show:
            print(f"\n   ↳ 직전 선택지 되묻기의 답으로 받음 → 기능을 {picked} 로 고정하고 원래 문장을 다시 처리합니다.")
    elif PENDING.get("kind") == "slot":
        cands = PENDING.get("options") or []
        q = text.replace(" ", "")
        hit = [c for c in cands if c.replace(" ", "") == q] or [c for c in cands if q and q in c.replace(" ", "")]
        if hit or (not cands and len(text) <= 12):
            carry, text_for_llm = "slot", None
            if show:
                print(f"\n   ↳ 직전 질문의 답으로 받음 → 모자랐던 값 '{PENDING['missing']}' 에 채웁니다.")

    # ② 의도분류 -------------------------------------------------------------
    if carry == "slot":
        call = dict(PENDING["call"])
        call[PENDING["missing"]] = text.strip()
        fn, d = call["fn"], {"route": "carry", "why": "직전 질문의 답을 채움", "margin": float("nan")}
        if show:
            print(f"\n② 의도분류   건너뜀 (직전 대화를 이어감)")
    else:
        t = time.perf_counter()
        p, ranked = classify(text if carry != "choice" else PENDING["text"])
        T["의도분류"] = (time.perf_counter() - t) * 1000
        d = decide(ranked)
        if show:
            print(f"\n② 의도분류   ({T['의도분류']:.1f} ms)")
            for i, (f, pr) in enumerate(ranked[:3]):
                bar = "█" * int(pr * 30)
                print(f"     {i + 1}등  {f:<13} {pr:6.3f}  {bar}")
            print(f"     1·2등 차이(margin) = {d['margin']:.3f}")
            route_ko = {"ask": "선택지로 되묻기", "skip": "바로 실행 (LLM 생략)", "llm": "LLM 으로"}[d["route"]]
            print(f"     판단: {route_ko}  ←  {d['why']}")

        if d["route"] == "ask" and carry != "choice":
            PENDING.clear()
            PENDING.update(kind="choice", opts=d["opts"], text=text)
            T["전체"] = (time.perf_counter() - t_all) * 1000
            if show:
                print(f"\n③~⑤ 건너뜀 — 실행하지 않습니다 (DB 를 바꾸지 않음)")
                print(f"\n⑥ 응답   🤖 {d['msg']}")
                if d["opts"]:
                    print(f"     선택지: {[CFG['choice_labels'][o] for o in d['opts']]}")
                print(f"     전체 {T['전체']:.0f} ms")
            return {"reply": d["msg"], "route": "ask", "options": d["opts"], "timings": T}

        # ③ function calling ----------------------------------------------
        if carry == "choice":
            fn = picked
            t = time.perf_counter()
            raw, cmean, cmin = call_llm(PENDING["text"])
            T["LLM"] = (time.perf_counter() - t) * 1000
            call = parse_json(raw)
            allowed = {"stop_items": {"store", "item"}, "complete": {"store", "receiver"},
                       "report_issue": {"store", "item", "issue", "qty", "unit"}}.get(fn, set())
            call = {"fn": fn, **{k: v for k, v in call.items() if k in allowed}}
            if show:
                print(f"\n③ function calling   ({T['LLM']:.0f} ms)")
                print(f"     LLM 원문  : {raw}")
                print(f"     고른 기능으로 맞춤 → {json.dumps(call, ensure_ascii=False)}")
                print(f"     LLM 확신도: 평균 {cmean:.3f} / 최솟값 {cmin:.3f}")
        elif d["route"] == "skip":
            fn, call = d["fn"], {"fn": d["fn"]}
            if show:
                print(f"\n③ function calling   건너뜀 — {d['why']}")
        else:
            t = time.perf_counter()
            raw, cmean, cmin = call_llm(text)
            T["LLM"] = (time.perf_counter() - t) * 1000
            call = parse_json(raw)
            fn = call.get("fn", d["fn"])
            if show:
                print(f"\n③ function calling   ({T['LLM']:.0f} ms)")
                print(f"     LLM 원문  : {raw}")
                print(f"     파싱 결과 : {json.dumps(call, ensure_ascii=False)}")
                flag = "  ← 낮음, 값 하나를 자신 없게 찍었을 수 있음" if cmin < 0.5 else ""
                print(f"     LLM 확신도: 평균 {cmean:.3f} / 최솟값 {cmin:.3f}{flag}")
    PENDING.clear()

    # ④ 값 보정 ---------------------------------------------------------------
    t = time.perf_counter()
    steps = []
    if fn in ("scan", "other"):
        call2, ask = dict(call), None
        call2["_stop"] = next_stop()
    else:
        call2, ask = resolve(call, steps)
    T["값 보정"] = (time.perf_counter() - t) * 1000
    if show:
        print(f"\n④ 값 보정   ({T['값 보정']:.1f} ms)")
        if not steps:
            print("     보정할 값 없음")
        for s in steps:
            cand = f"   후보 {s['후보']}" if len(s["후보"]) > 1 else ""
            print(f"     {s['값']:<10} {s['말한 것']}  →  {s['찾은 것']}   [{s['방법']} {s['점수']}]{cand}")
        if ask:
            print(f"     ✋ 값이 모자라 실행하지 않습니다: {ask}")

    if ask:
        miss = "item" if "품목" in ask else ("issue" if "문제" in ask else "store")
        PENDING.update(kind="slot", call={k: v for k, v in call2.items() if k != "_stop"},
                       missing=miss, options=[s["후보"] for s in steps if len(s["후보"]) > 1][:1] and
                       [c for s in steps if len(s["후보"]) > 1 for c in s["후보"]] or [])
        T["전체"] = (time.perf_counter() - t_all) * 1000
        if show:
            print(f"\n⑤ 실행   건너뜀 (DB 를 바꾸지 않음)")
            print(f"\n⑥ 응답   🤖 {ask}")
            print(f"     전체 {T['전체']:.0f} ms")
        return {"reply": ask, "route": "ask_slot", "options": PENDING["options"], "timings": T}

    # ⑤ 실행 -------------------------------------------------------------------
    req = build_request(fn, call2)
    if show:
        print(f"\n⑤ 실행   백엔드에 보낼 요청")
        if req["method"] is None:
            print(f"     없음 — {fn} 은 DB 를 바꾸지 않습니다")
        else:
            print(f"     {req['method']} {BACKEND_URL}{req['path']}")
            if req["body"] is not None:
                print(f"     body: {json.dumps(req['body'], ensure_ascii=False)}")
        print(f"     보내는 중? {send}  ({'실제 API 호출' if send else 'SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리'})")
    t = time.perf_counter()
    if send and req["method"]:
        try:
            result = send_real(req)
        except Exception as e:
            print(f"     [실패] {type(e).__name__}: {e} → 가상 DB 로 대신 처리합니다")
            send, result = False, run_virtual(fn, call2, req)
    else:
        send, result = False, run_virtual(fn, call2, req)
    T["실행"] = (time.perf_counter() - t) * 1000

    # ⑥ 응답 -------------------------------------------------------------------
    reply = make_reply(fn, call2, result, send)
    T["전체"] = (time.perf_counter() - t_all) * 1000
    if show:
        print(f"\n⑥ 응답   🤖 {reply}")
        print("     단계별 시간(ms): " + " · ".join(f"{k} {v:.1f}" for k, v in T.items()))
    return {"reply": reply, "route": d["route"], "fn": fn, "timings": T}

## 5. 예시 문장 돌려 보기

기능 6개를 모두 포함하고, 값 되묻기 · 선택지 되묻기 · DB 점포 이름 · 잡담까지 넣었습니다.

In [9]:
EXAMPLES = [
    "다음 어디야",                      # next_stop
    "여기 뭐 내려야 돼",                 # stop_items (점포를 말하지 않음 -> 다음 배송지)
    "서울숲점 콜라 몇 박스야",            # stop_items + DB 점포 이름
    "우유 두 개 터졌어",                 # report_issue ("우유" 는 별칭으로 서울우유에 걸림)
    "점장님한테 넘겼어",                 # complete -> 다음 배송지가 옮겨감
    "물건 확인할게요",                   # scan 이 정답인 문장
    "몇 개 남았지",                     # next_stop / stop_items 로 헷갈리는 문장
    "노래 틀어줘",                      # other (LLM 생략)
    "지금 몇 시야",                     # other 인데 의도분류가 헷갈림 -> LLM 이 바로잡음
    "이거 미지근해",                    # report_issue, 품목을 안 말함 -> 값 되묻기
    "바나나우유",                       #   바로 위 질문의 답
    "다음 거 알려줘",                   # next_stop
]
for s in EXAMPLES:
    run(s)
print("=" * 100)

① 입력   🗣  다음 어디야



② 의도분류   (184.6 ms)
     1등  next_stop      0.997  █████████████████████████████
     2등  complete       0.001  
     3등  other          0.001  
     1·2등 차이(margin) = 0.996
     판단: LLM 으로  ←  next_stop 은 값을 뽑아야 하므로 LLM 으로


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



③ function calling   (559 ms)
     LLM 원문  : {"fn":"next_stop"}
     파싱 결과 : {"fn": "next_stop"}
     LLM 확신도: 평균 1.000 / 최솟값 0.999

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  GS25 성수연무장점   [다음 배송지로 채움 —]

⑤ 실행   백엔드에 보낼 요청
     GET http://127.0.0.1:8000/api/routes/today
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 다음은 GS25 성수연무장점이에요. 남은 곳은 3곳이에요.
     단계별 시간(ms): 의도분류 184.6 · LLM 558.7 · 값 보정 0.0 · 실행 0.0 · 전체 744.4
① 입력   🗣  여기 뭐 내려야 돼

② 의도분류   (6.8 ms)
     1등  stop_items     0.996  █████████████████████████████
     2등  complete       0.001  
     3등  scan           0.001  
     1·2등 차이(margin) = 0.996
     판단: LLM 으로  ←  stop_items 은 값을 뽑아야 하므로 LLM 으로



③ function calling   (362 ms)
     LLM 원문  : {"fn":"stop_items"}
     파싱 결과 : {"fn": "stop_items"}
     LLM 확신도: 평균 1.000 / 최솟값 0.999

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  GS25 성수연무장점   [다음 배송지로 채움 —]

⑤ 실행   백엔드에 보낼 요청
     GET http://127.0.0.1:8000/api/stops/1
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 GS25 성수연무장점 납품은 서울우유 12개, 바나나우유 6개, 제육도시락 4개, 생수 2박스예요.
     단계별 시간(ms): 의도분류 6.8 · LLM 361.6 · 값 보정 0.0 · 실행 0.0 · 전체 369.0
① 입력   🗣  서울숲점 콜라 몇 박스야

② 의도분류   (6.6 ms)
     1등  stop_items     0.997  █████████████████████████████
     2등  report_issue   0.001  
     3등  complete       0.001  
     1·2등 차이(margin) = 0.996
     판단: LLM 으로  ←  stop_items 은 값을 뽑아야 하므로 LLM 으로


...


③ function calling   (362 ms)
     LLM 원문  : {"fn":"stop_items"}
     파싱 결과 : {"fn": "stop_items"}
     LLM 확신도: 평균 0.998 / 최솟값 0.990

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  CU 서울숲점   [다음 배송지로 채움 —]

⑤ 실행   백엔드에 보낼 요청
     GET http://127.0.0.1:8000/api/stops/2
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 CU 서울숲점 납품은 서울우유 6개, 참치삼각김밥 10개, 콜라 1박스예요.
     단계별 시간(ms): 의도분류 5.8 · LLM 361.9 · 값 보정 0.0 · 실행 0.0 · 전체 368.2
① 입력   🗣  몇 개 남았지

② 의도분류   (5.3 ms)
     1등  next_stop      0.997  █████████████████████████████
     2등  other          0.001  
     3등  complete       0.001  
     1·2등 차이(margin) = 0.996
     판단: LLM 으로  ←  next_stop 은 값을 뽑아야 하므로 LLM 으로



③ function calling   (358 ms)
     LLM 원문  : {"fn":"stop_items"}
     파싱 결과 : {"fn": "stop_items"}
     LLM 확신도: 평균 0.973 / 최솟값 0.817

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  CU 서울숲점   [다음 배송지로 채움 —]

⑤ 실행   백엔드에 보낼 요청
     GET http://127.0.0.1:8000/api/stops/2
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 CU 서울숲점 납품은 서울우유 6개, 참치삼각김밥 10개, 콜라 1박스예요.
     단계별 시간(ms): 의도분류 5.3 · LLM 357.5 · 값 보정 0.0 · 실행 0.0 · 전체 363.3
① 입력   🗣  노래 틀어줘

② 의도분류   (5.6 ms)
     1등  other          0.996  █████████████████████████████
     2등  report_issue   0.001  
     3등  next_stop      0.001  
     1·2등 차이(margin) = 0.995
     판단: 바로 실행 (LLM 생략)  ←  other 은 값이 필요 없고 확률 0.996 >= 0.9 이라 LLM 생략

③ function calling   건너뜀 — other 은 값이 필요 없고 확률 0.996 >= 0.9 이라 LLM 생략

④ 값 보정   (0.0 ms)
     보정할 값 없음

⑤ 실행   백엔드에 보낼 요청
     없음 — other 은 DB 를 바꾸지 않습니다
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 죄송해요, 배송 업무만 도와드릴 수 있어요.
     단계별 시간(ms): 의도분류 5.6 · 값 보정 


③ function calling   (310 ms)
     LLM 원문  : {"fn":"other"}
     파싱 결과 : {"fn": "other"}
     LLM 확신도: 평균 1.000 / 최솟값 0.999

④ 값 보정   (0.0 ms)
     보정할 값 없음

⑤ 실행   백엔드에 보낼 요청
     없음 — other 은 DB 를 바꾸지 않습니다
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 죄송해요, 배송 업무만 도와드릴 수 있어요.
     단계별 시간(ms): 의도분류 5.1 · LLM 309.9 · 값 보정 0.0 · 실행 0.0 · 전체 315.5
① 입력   🗣  이거 미지근해

② 의도분류   (5.7 ms)
     1등  report_issue   0.996  █████████████████████████████
     2등  other          0.001  
     3등  stop_items     0.001  
     1·2등 차이(margin) = 0.994
     판단: LLM 으로  ←  report_issue 은 값을 뽑아야 하므로 LLM 으로



③ function calling   (704 ms)
     LLM 원문  : {"fn":"report_issue","issue":"온도이상"}
     파싱 결과 : {"fn": "report_issue", "issue": "온도이상"}
     LLM 확신도: 평균 0.998 / 최솟값 0.971

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  CU 서울숲점   [다음 배송지로 채움 —]
     ✋ 값이 모자라 실행하지 않습니다: 어떤 품목인가요?

⑤ 실행   건너뜀 (DB 를 바꾸지 않음)

⑥ 응답   🤖 어떤 품목인가요?
     전체 710 ms
① 입력   🗣  바나나우유

   ↳ 직전 질문의 답으로 받음 → 모자랐던 값 'item' 에 채웁니다.

② 의도분류   건너뜀 (직전 대화를 이어감)

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  CU 서울숲점   [다음 배송지로 채움 —]
     item       바나나우유  →  바나나우유   [정확 1.0]

⑤ 실행   백엔드에 보낼 요청
     POST http://127.0.0.1:8000/api/stops/2/issues
     body: {"type": "온도이상", "source": "voice", "product_id": 2}
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 CU 서울숲점 바나나우유 온도이상으로 기록했어요.
     단계별 시간(ms): 값 보정 0.0 · 실행 0.0 · 전체 0.1
① 입력   🗣  다음 거 알려줘

② 의도분류   (5.7 ms)
     1등  next_stop      0.997  █████████████████████████████
     2등  complete       0.001  
     3등  stop_items     0.001  
     1·2등 차이(mar


③ function calling   (360 ms)
     LLM 원문  : {"fn":"next_stop"}
     파싱 결과 : {"fn": "next_stop"}
     LLM 확신도: 평균 1.000 / 최솟값 0.999

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  CU 서울숲점   [다음 배송지로 채움 —]

⑤ 실행   백엔드에 보낼 요청
     GET http://127.0.0.1:8000/api/routes/today
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 다음은 CU 서울숲점이에요. 남은 곳은 2곳이에요.
     단계별 시간(ms): 의도분류 5.7 · LLM 360.3 · 값 보정 0.0 · 실행 0.0 · 전체 366.6


### 위 결과에서 볼 것

- **"여기 뭐 내려야 돼"** — 점포를 말하지 않아 **다음 배송지**로 채웠습니다.
- **"우유 두 개 터졌어"** — `seed.py` 에 `우유` 가 **서울우유의 별칭**으로 등록돼 있어 되묻지 않고 바로 정해집니다.
  (품목을 아예 말하지 않은 **"이거 미지근해"** 에서 값 되묻기가 걸립니다.)
- **"점장님한테 넘겼어"** 다음부터 **다음 배송지가 CU 서울숲점으로 옮겨갑니다.** 완료 처리가 실제로 반영된 것입니다.
- **"지금 몇 시야"** — 의도분류는 `next_stop` 으로 헷갈렸지만(0.849) **LLM 이 `other` 로 바로잡았습니다.**
  v2.1 에서 `next_stop` 을 LLM 건너뛰기 대상에서 뺀 이유가 이것입니다. 예전 설정이었다면 그냥 다음 배송지를 읽어 줬을 것입니다.
- **"물건 확인할게요"** — 정답은 `scan` 인데 `stop_items` 로 나왔습니다. **지금 모델의 알려진 약점**입니다
  (`docs/1st_report_summary.md` 의 "원인 B"). 숨기지 않고 그대로 둡니다.
- **되묻기가 거의 안 걸립니다.** 모델이 확신하기 때문이고, test 431문장 중 되물은 것은 2건이었습니다.

### 선택지 되묻기 이어받기

위 예시에서 선택지 되묻기가 걸리지 않았다면, **일부러 헷갈리게 만든 확률**로 그 흐름만 따로 보여 줍니다.
(모델이 확신하면 되묻지 않는 것이 정상입니다. test 431문장 중 되묻은 것은 2건뿐이었습니다.)

In [10]:
PENDING.clear()
PENDING.update(kind="choice", opts=["next_stop", "stop_items"], text="다음 거 알려줘")
print('🤖 (앞선 질문) 남은 배송지 말씀하시는 건가요, 내릴 물건 말씀하시는 건가요?')
run("배송지")

PENDING.clear()
PENDING.update(kind="choice", opts=["next_stop", "stop_items"], text="다음 거 알려줘")
print('\n🤖 (앞선 질문) 남은 배송지 말씀하시는 건가요, 내릴 물건 말씀하시는 건가요?')
run("두 번째")

🤖 (앞선 질문) 남은 배송지 말씀하시는 건가요, 내릴 물건 말씀하시는 건가요?
① 입력   🗣  배송지

   ↳ 직전 선택지 되묻기의 답으로 받음 → 기능을 next_stop 로 고정하고 원래 문장을 다시 처리합니다.

② 의도분류   (6.2 ms)
     1등  next_stop      0.997  █████████████████████████████
     2등  complete       0.001  
     3등  stop_items     0.001  
     1·2등 차이(margin) = 0.996
     판단: LLM 으로  ←  next_stop 은 값을 뽑아야 하므로 LLM 으로



③ function calling   (358 ms)
     LLM 원문  : {"fn":"next_stop"}
     고른 기능으로 맞춤 → {"fn": "next_stop"}
     LLM 확신도: 평균 1.000 / 최솟값 0.999

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  CU 서울숲점   [다음 배송지로 채움 —]

⑤ 실행   백엔드에 보낼 요청
     GET http://127.0.0.1:8000/api/routes/today
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 다음은 CU 서울숲점이에요. 남은 곳은 2곳이에요.
     단계별 시간(ms): 의도분류 6.2 · LLM 358.0 · 값 보정 0.0 · 실행 0.0 · 전체 364.7

🤖 (앞선 질문) 남은 배송지 말씀하시는 건가요, 내릴 물건 말씀하시는 건가요?
① 입력   🗣  두 번째

   ↳ 직전 선택지 되묻기의 답으로 받음 → 기능을 stop_items 로 고정하고 원래 문장을 다시 처리합니다.

② 의도분류   (5.3 ms)
     1등  next_stop      0.997  █████████████████████████████
     2등  complete       0.001  
     3등  stop_items     0.001  
     1·2등 차이(margin) = 0.996
     판단: LLM 으로  ←  next_stop 은 값을 뽑아야 하므로 LLM 으로



③ function calling   (357 ms)
     LLM 원문  : {"fn":"next_stop"}
     고른 기능으로 맞춤 → {"fn": "stop_items"}
     LLM 확신도: 평균 1.000 / 최솟값 0.999

④ 값 보정   (0.0 ms)
     store      (말하지 않음)  →  CU 서울숲점   [다음 배송지로 채움 —]

⑤ 실행   백엔드에 보낼 요청
     GET http://127.0.0.1:8000/api/stops/2
     보내는 중? False  (SEND_TO_BACKEND=False 라 노트북 안 가상 DB 로 처리)

⑥ 응답   🤖 CU 서울숲점 납품은 서울우유 6개, 참치삼각김밥 10개, 콜라 1박스예요.
     단계별 시간(ms): 의도분류 5.3 · LLM 356.7 · 값 보정 0.0 · 실행 0.0 · 전체 362.6


{'reply': 'CU 서울숲점 납품은 서울우유 6개, 참치삼각김밥 10개, 콜라 1박스예요.',
 'route': 'llm',
 'fn': 'stop_items',
 'timings': {'의도분류': 5.345100536942482,
  'LLM': 356.68820049613714,
  '값 보정': 0.009600073099136353,
  '실행': 0.0019995495676994324,
  '전체': 362.55429964512587}}

### 오늘 운행 상태 (위 예시로 바뀐 결과)

`complete` 와 `report_issue` 는 실제로 상태를 바꿉니다. 위에서 "점장님한테 넘겼어" 를 처리했으므로
첫 배송지가 완료로 바뀌고 **다음 배송지가 자동으로 옮겨간** 것을 볼 수 있습니다.

In [11]:
for s in STOPS:
    mark = "완료" if s["status"] == "done" else "대기"
    iss = "".join(f"\n        이상: {i['type']} {i.get('product_name') or ''} "
                  f"{i.get('qty') or ''}{i.get('unit') or ''} (source={i['source']})" for i in s["issues"])
    print(f"  {s['seq']}. {s['store']['name']:<18} {mark}"
          f"{'  인수자 ' + s['receiver'] if s['receiver'] else ''}{iss}")
nxt = next_stop()
print("\n다음 배송지:", nxt["store"]["name"] if nxt else "(오늘 배송 끝)")

  1. GS25 성수연무장점        완료  인수자 점장님
        이상: 파손 서울우유 2개 (source=voice)
  2. CU 서울숲점            대기
        이상: 온도이상 바나나우유  (source=voice)
  3. 세븐일레븐 뚝섬점          대기

다음 배송지: CU 서울숲점


## 6. 직접 입력해 보기

`RUN_INTERACTIVE = True` 로 바꾸고 이 셀을 실행하면 직접 말을 넣어 볼 수 있습니다. 빈 줄이나 `그만` 을 넣으면 끝납니다.
(깃헙에 올라간 출력에는 이 셀이 실행되지 않은 상태로 남습니다.)

In [12]:
RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    print("문장을 입력하세요. 빈 줄 또는 '그만' 이면 끝납니다.")
    while True:
        try:
            s = input("🗣  ").strip()
        except EOFError:
            break
        if not s or s in ("그만", "종료", "quit", "exit"):
            break
        run(s)
else:
    print("RUN_INTERACTIVE = True 로 바꾸면 직접 입력해 볼 수 있습니다.")

RUN_INTERACTIVE = True 로 바꾸면 직접 입력해 볼 수 있습니다.


---

## 더 볼 것

| | |
| --- | --- |
| 구조 그림 | [`docs/architecture_v2_1.png`](../docs/architecture.png) |
| 1차 보고 정리 | [`docs/1st_report_summary.md`](../docs/1st_report_summary.md) |
| 백엔드 연결 방법 | [`docs/backend_integration.md`](../docs/backend_integration.md) |
| 모델 만드는 노트북 | [`LogiOn_AI_pipeline.ipynb`](LogiOn_AI_pipeline.ipynb) |
| 설정 한 파일 | [`config/pipeline_v2_1.json`](../config/pipeline_v2_1.json) |
| 측정 결과 | [`results/v2_1/`](../results/v2_1/) |

**주의**: 여기 나오는 시간(ms)은 **이 PC** 에서 잰 값이고 보드(RK3588) 값이 아닙니다. STT·TTS 단계는 아직 포함돼 있지 않습니다.